# Module A — Conversion BDD100K + Entraînement YOLOv11l
## Scénario : Autoroute — Version Kaggle (2x T4)

**Ce notebook fait tout en une seule fois :**
1. Installe les dépendances
2. Copie le dataset BDD100K Highway
3. Convertit les labels BDD100K JSON → format YOLO .txt
4. Crée le fichier dataset.yaml
5. Entraîne **YOLOv11l** (25.3M params, 86.9 GFLOPs) sur les 2 GPU T4
6. Sauvegarde le meilleur modèle

**Prérequis :**
- Dataset `bdd100k_highway/` disponible dans Kaggle
- Activer GPU dans Kaggle : Settings → Accelerator → **GPU T4 x2**
- Temps estimé : **~13–17 h** (peut nécessiter 2 sessions Kaggle de 12h)

**Structure attendue :**
```
bdd100k_highway/
├── images/
│   ├── train/   (17414 images .jpg)
│   └── val/     (2499 images .jpg)
└── labels/
    ├── train/   (17414 fichiers .json)
    └── val/     (2499 fichiers .json)
```

**💡 Si la session Kaggle coupe avant la fin** — relancer la cellule 5 avec `resume=True` :
```python
model = YOLO('/kaggle/working/runs/detect/yolov11l_autoroute/weights/last.pt')
results = model.train(resume=True)
```

In [ ]:
import shutil
shutil.copytree('/kaggle/input/datasets/benkhalifamahmoud/bdd100k-highway/bdd100k_highway', '/kaggle/working/bdd100k-highway/', dirs_exist_ok=True)

In [ ]:
# ============================================================
# CELLULE 1 — Installation et vérification GPU
# ============================================================
!pip install ultralytics gdown -q

import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
else:
    print('ATTENTION : Pas de GPU !')
    print('Aller dans Settings → Accelerator → GPU T4 x2')

In [ ]:
import os
import json
import shutil

# Source paths
TRAIN_LBL = '/kaggle/working/bdd100k-highway/labels/train'
TRAIN_IMG = '/kaggle/working/bdd100k-highway/images/train'
VAL_LBL   = '/kaggle/working/bdd100k-highway/labels/val'
VAL_IMG   = '/kaggle/working/bdd100k-highway/images/val'
# ============================================================
# CELLULE 3 — Conversion BDD100K JSON → YOLO .txt
# ============================================================
import json
import shutil

CLASS_MAP = {
    'car':           0,
    'truck':         1,
    'bus':           2,
    'pedestrian':    3,
    'rider':         4,
    'traffic sign':  5,
    'traffic light': 6
}

IMG_W, IMG_H = 1280, 720

OUT_TRAIN_LBL = '/kaggle/working/dataset/labels/train'
OUT_VAL_LBL   = '/kaggle/working/dataset/labels/val'
OUT_TRAIN_IMG = '/kaggle/working/dataset/images/train'
OUT_VAL_IMG   = '/kaggle/working/dataset/images/val'

for folder in [OUT_TRAIN_LBL, OUT_VAL_LBL, OUT_TRAIN_IMG, OUT_VAL_IMG]:
    os.makedirs(folder, exist_ok=True)


def convert_split(lbl_dir, img_src_dir, out_lbl_dir, out_img_dir, split):
    json_files = [f for f in os.listdir(lbl_dir) if f.endswith('.json')]
    converted, skipped = 0, 0

    print(f'Conversion {split} : {len(json_files)} fichiers...')

    for i, jf in enumerate(json_files):
        with open(os.path.join(lbl_dir, jf), 'r') as f:
            data = json.load(f)

        frames = data.get('frames', [])
        if not frames:
            skipped += 1
            continue
        objects = frames[0].get('objects', [])

        yolo_lines = []
        for obj in objects:
            cat = obj.get('category', '')
            if cat not in CLASS_MAP:
                continue
            box = obj.get('box2d')
            if not box:
                continue

            x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
            x_center = min(max((x1 + x2) / (2 * IMG_W), 0), 1)
            y_center = min(max((y1 + y2) / (2 * IMG_H), 0), 1)
            width    = min(max((x2 - x1) / IMG_W, 0), 1)
            height   = min(max((y2 - y1) / IMG_H, 0), 1)

            yolo_lines.append(
                f"{CLASS_MAP[cat]} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        if not yolo_lines:
            skipped += 1
            continue

        base_name = jf.replace('.json', '')
        img_name  = base_name + '.jpg'
        src_img   = os.path.join(img_src_dir, img_name)

        if not os.path.exists(src_img):
            skipped += 1
            continue

        txt_path = os.path.join(out_lbl_dir, base_name + '.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

        shutil.copy2(src_img, os.path.join(out_img_dir, img_name))
        converted += 1

        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(json_files)}...')

    print(f'{split.upper()} — Convertis : {converted} | Ignorés : {skipped}')
    return converted


print('=== Conversion TRAIN ===')
n_train = convert_split(TRAIN_LBL, TRAIN_IMG, OUT_TRAIN_LBL, OUT_TRAIN_IMG, 'train')

print('\n=== Conversion VAL ===')
n_val = convert_split(VAL_LBL, VAL_IMG, OUT_VAL_LBL, OUT_VAL_IMG, 'val')

print(f'\nDataset converti : {n_train} train | {n_val} val')

In [ ]:
# ============================================================
# CELLULE 4 — Création du fichier dataset.yaml
# ============================================================
yaml_content = """# Dataset BDD100K — Scénario Autoroute
# Filtré sur scene=highway | 7 classes

path: /kaggle/working/dataset
train: images/train
val: images/val

nc: 7

names:
  0: car
  1: truck
  2: bus
  3: pedestrian
  4: rider
  5: traffic sign
  6: traffic light
"""

YAML_PATH = '/kaggle/working/dataset_autoroute.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('dataset_autoroute.yaml créé')
print(yaml_content)

In [ ]:
# ============================================================
# CELLULE 5 — Entraînement YOLOv11l (2x T4)
# ============================================================
# Config optimisée pour Kaggle 2x Tesla T4 (16GB chacune)
# YOLOv11l : 25.3M paramètres, 86.9 GFLOPs
# Temps estimé : ~13-17h en imgsz=1280 batch=6
# ⚠️ Kaggle limite les sessions à 12h — si besoin, reprendre avec resume=True

from ultralytics import YOLO
print('=== Entraînement YOLOv11l — Scénario Autoroute (2x T4) ===')

model = YOLO('yolo11l.pt')

results = model.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=1280,
    batch=6,              # 3 par GPU — ~14 GB VRAM/GPU
    device=[0, 1],        # Utilise les 2 GPU T4 (DDP)
    name='yolov11l_autoroute',
    # Augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    translate=0.1,
    scale=0.5,
    degrees=0.0,
    # Optimisation mémoire & vitesse
    amp=True,             # Mixed precision (obligatoire sur T4)
    cache=False,          # True si assez de RAM, sinon False
    workers=4,
    # Sauvegarde
    save_period=5,        # Checkpoint tous les 5 epochs (utile si session coupée)
    patience=10,          # Early stopping si pas d'amélioration pendant 10 epochs
    plots=True,
    verbose=True
)

# Récupération des métriques
map50   = results.results_dict.get('metrics/mAP50(B)', 0)
map5095 = results.results_dict.get('metrics/mAP50-95(B)', 0)
prec    = results.results_dict.get('metrics/precision(B)', 0)
rec     = results.results_dict.get('metrics/recall(B)', 0)

print(f'mAP@0.5      : {map50:.4f}')
print(f'mAP@0.5:0.95 : {map5095:.4f}')
print(f'Précision    : {prec:.4f}')
print(f'Rappel       : {rec:.4f}')

In [ ]:
# ============================================================
# CELLULE 6 — Résumé des métriques & sauvegarde du modèle
# ============================================================
import shutil
import pandas as pd
import matplotlib.pyplot as plt

# Tableau récapitulatif
df = pd.DataFrame({
    'Métrique': ['mAP@0.5', 'mAP@0.5:0.95', 'Précision', 'Rappel', 'Paramètres', 'GFLOPs'],
    'YOLOv11l': [f'{map50:.4f}', f'{map5095:.4f}', f'{prec:.4f}', f'{rec:.4f}', '25.3M', '86.9'],
})
print(df.to_string(index=False))
df.to_csv('/kaggle/working/yolov11l_metrics.csv', index=False)

# Graphique des métriques
metrics = ['mAP@0.5', 'mAP@0.5:0.95', 'Précision', 'Rappel']
vals    = [map50, map5095, prec, rec]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metrics, vals, color='#D85A30')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('YOLOv11l — Scénario Autoroute', fontsize=13)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/kaggle/working/yolov11l_metrics.png', dpi=150)
plt.show()

# Sauvegarder le meilleur modèle pour téléchargement
best_pt = '/kaggle/working/runs/detect/yolov11l_autoroute/weights/best.pt'
shutil.copy(best_pt, '/kaggle/working/best_model_autoroute.pt')
print(f'\nmAP@0.5 = {map50:.4f}')
print('Modèle sauvegardé : /kaggle/working/best_model_autoroute.pt')
print('Pour télécharger : panneau Output à droite → best_model_autoroute.pt → Download')